# Logging and alerting for the self-organising map

This notebook follows on from [kohonen_modified.ipynb](kohonen_modified.ipynb). That notebook made Sam's code fast, general, validated, reusable and measurable. This one answers the question the panel asked at the end: if the map ran behind a website, how would you know it was working, other than with print statements?

The short answer has three parts. Every event is logged with a level, a time and a source. Anything serious reaches a person, not just a file. The service reports its own health, including whether the map still fits the data.

**Steps**

1. Why logging and not print
2. Log from inside the class
3. Structured logs for a log store
4. Alerting: errors reach a person
5. Health check with quantisation error
6. A FastAPI endpoint with request logging
7. Tests that check the logging
8. What this looks like in production

Each step states the problem, the fix, and shows the result. Everything up to step 5 is the standard library only. Step 6 needs `fastapi` and `httpx`.

## Starting point: the `SOM` class

This is the class from the end of `kohonen_modified.ipynb`, unchanged. It trains, predicts, saves, loads and measures quantisation error. It is also silent: it tells you nothing while it runs.

The demo data is 200 random colours instead of Sam's 10, with 50 more kept aside. A map trained on 10 points fits those 10 points very closely, so its error says little about new data. Step 5 needs a reference that generalises.

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt


class SOM:
    """
    Self-organising map.

    Usage:
        som = SOM(width=10, height=10, seed=42)
        som.fit(data, n_iterations=100)
        positions = som.predict(new_data)
    """

    def __init__(self, width, height, α0=0.1, seed=None):
        if max(width, height) <= 2:
            raise ValueError("Largest grid side must be > 2, otherwise the radius cannot decay")
        self.width = width
        self.height = height
        self.α0 = α0
        self.seed = seed
        self.weights = None

    def fit(self, input_data, n_iterations):
        """Train the map on input_data (n_samples, n_features)."""
        if input_data.ndim != 2:
            raise ValueError("input_data must be a 2D array: (n_samples, n_features)")
        if n_iterations < 1:
            raise ValueError("n_iterations must be at least 1")

        rng = np.random.default_rng(self.seed)
        self.weights = rng.random((self.width, self.height, input_data.shape[1]))

        σ0 = max(self.width, self.height) / 2
        λ = n_iterations / np.log(σ0)
        gx, gy = np.indices((self.width, self.height))

        for t in range(n_iterations):
            σt = σ0 * np.exp(-t / λ)
            αt = self.α0 * np.exp(-t / λ)
            for vt in input_data:
                bmu_x, bmu_y = self._bmu(vt)
                d2 = (gx - bmu_x) ** 2 + (gy - bmu_y) ** 2
                θt = np.exp(-d2 / (2 * σt ** 2))
                self.weights += αt * θt[..., None] * (vt - self.weights)
        return self

    def _bmu(self, vt):
        """Grid position of the node closest to vt."""
        bmu = np.argmin(np.sum((self.weights - vt) ** 2, axis=2))
        return np.unravel_index(bmu, (self.width, self.height))

    def predict(self, input_data):
        """Map each input to its BMU grid position."""
        if self.weights is None:
            raise RuntimeError("Call fit() before predict()")
        return np.array([self._bmu(vt) for vt in input_data])

    def quantisation_error(self, input_data):
        """Average distance from each input to its BMU. Lower = map fits the data better."""
        if self.weights is None:
            raise RuntimeError("Call fit() before quantisation_error()")
        errors = [np.linalg.norm(self.weights[self._bmu(vt)] - vt) for vt in input_data]
        return np.mean(errors)

    def save(self, path):
        """Save weights and settings to a .npz file."""
        np.savez(path, weights=self.weights, width=self.width, height=self.height,
                 α0=self.α0, seed=-1 if self.seed is None else self.seed)

    @classmethod
    def load(cls, path):
        """Load a saved map."""
        f = np.load(path)
        seed = int(f["seed"])
        som = cls(int(f["width"]), int(f["height"]), float(f["α0"]),
                  None if seed == -1 else seed)
        som.weights = f["weights"]
        return som


rng = np.random.default_rng(0)
data = rng.random((200, 3))        # 200 random colours rather than Sam's 10, so the error below means something
held_out = rng.random((50, 3))     # kept aside: the reference for the health check in step 5
som = SOM(width=10, height=10, seed=42).fit(data, n_iterations=200)
print("Trained. Quantisation error on training data:", round(som.quantisation_error(data), 4))

Trained. Quantisation error on training data: 0.1366


## Step 1: Why logging and not print

**Problem**

A print statement goes to the screen of whoever ran the code. Behind a website nobody is watching that screen. A print has no timestamp, no severity, no record of which module wrote it, and it cannot be switched off or sent somewhere else without editing the code.

**Fix**

The standard library `logging` module. Each message carries a level, a time and the name of the logger that wrote it. Handlers decide where messages go (screen, file, log store, alert). The level decides who needs to see it.

| Level | What it means for the SOM | Who sees it |
|---|---|---|
| `DEBUG` | Per-iteration radius and learning rate | Nobody, unless they turn it on to investigate |
| `INFO` | Training started and finished, model saved, a request served | The log store, searched when something is wrong |
| `WARNING` | Quantisation error above the threshold, a slow request, a client sent bad input | A dashboard, and a daily review |
| `ERROR` | A prediction failed for an unexpected reason | An on-call person, within minutes |
| `CRITICAL` | The service cannot load the model at startup | A page to the on-call person, now |

**How**

One named logger for the whole package. Handlers attach to it once. The code only ever calls `logger.info(...)` and friends and never decides where the message goes. That is the handler's job.

In [2]:
import logging
import sys

logger = logging.getLogger("som")
logger.setLevel(logging.DEBUG)
logger.propagate = False          # keep the notebook's root logger out of it


def reset_handlers():
    """Start each step with a clean set of handlers, so output is not duplicated."""
    for h in list(logger.handlers):
        logger.removeHandler(h)


reset_handlers()
console = logging.StreamHandler(sys.stdout)
console.setFormatter(logging.Formatter("%(asctime)s %(levelname)-8s %(name)s: %(message)s"))
logger.addHandler(console)

logger.debug("per-iteration detail, hidden in production")
logger.info("the map finished training")
logger.warning("quantisation error is above the threshold")
logger.error("a prediction failed")
logger.critical("the model file is missing")

2026-10-30 11:53:14,453 DEBUG    som: per-iteration detail, hidden in production


2026-10-30 11:53:14,453 INFO     som: the map finished training


2026-10-30 11:53:14,454 WARNING  som: quantisation error is above the threshold


2026-10-30 11:53:14,454 ERROR    som: a prediction failed


2026-10-30 11:53:14,454 CRITICAL som: the model file is missing


## Step 2: Log from inside the class

**Problem**

The class is silent. When training runs as a batch job at 2 am, nobody knows whether it started, how long it took, or whether it was any good.

**Fix**

Log at the points that matter:

- `fit` start: grid size, number of features, number of iterations, at `INFO`
- `fit` end: duration and final quantisation error, at `INFO`
- every tenth of the run: iteration, radius and learning rate, at `DEBUG`
- `save` and `load`: the path, at `INFO`
- every validation failure: at `ERROR`, then raise. Log and raise, never log and swallow. The caller still gets the exception, and the log store has the record.

**How**

The class uses a module-level logger, `logging.getLogger(__name__)`, which here resolves to the `som` logger set up above. In a package that name becomes `som.model`, so logs can be filtered by module. The lines marked `# LOGGING` are the only change.

In [3]:
class SOM(SOM):
    """The same SOM, with logging at the points that matter."""

    def __init__(self, width, height, α0=0.1, seed=None):
        if max(width, height) <= 2:
            logger.error("Invalid grid %sx%s: largest side must be > 2", width, height)   # LOGGING
            raise ValueError("Largest grid side must be > 2, otherwise the radius cannot decay")
        super().__init__(width, height, α0, seed)

    def fit(self, input_data, n_iterations):
        if input_data.ndim != 2:
            logger.error("Invalid input shape %s: expected (n_samples, n_features)", input_data.shape)  # LOGGING
            raise ValueError("input_data must be a 2D array: (n_samples, n_features)")
        if n_iterations < 1:
            logger.error("Invalid n_iterations=%s", n_iterations)   # LOGGING
            raise ValueError("n_iterations must be at least 1")

        logger.info("fit started: grid=%sx%s features=%s samples=%s iterations=%s",   # LOGGING
                    self.width, self.height, input_data.shape[1], input_data.shape[0], n_iterations)
        start = time.perf_counter()

        rng = np.random.default_rng(self.seed)
        self.weights = rng.random((self.width, self.height, input_data.shape[1]))
        σ0 = max(self.width, self.height) / 2
        λ = n_iterations / np.log(σ0)
        gx, gy = np.indices((self.width, self.height))
        report_every = max(1, n_iterations // 10)

        for t in range(n_iterations):
            σt = σ0 * np.exp(-t / λ)
            αt = self.α0 * np.exp(-t / λ)
            if t % report_every == 0:
                logger.debug("iteration=%s radius=%.3f learning_rate=%.4f", t, σt, αt)   # LOGGING
            for vt in input_data:
                bmu_x, bmu_y = self._bmu(vt)
                d2 = (gx - bmu_x) ** 2 + (gy - bmu_y) ** 2
                θt = np.exp(-d2 / (2 * σt ** 2))
                self.weights += αt * θt[..., None] * (vt - self.weights)

        self.train_error_ = self.quantisation_error(input_data)
        logger.info("fit finished in %.2fs: quantisation_error=%.4f",   # LOGGING
                    time.perf_counter() - start, self.train_error_)
        return self

    def save(self, path):
        super().save(path)
        logger.info("model saved to %s", path)   # LOGGING

    @classmethod
    def load(cls, path):
        som = super().load(path)
        logger.info("model loaded from %s", path)   # LOGGING
        return som

In [4]:
som = SOM(width=10, height=10, seed=42).fit(data, n_iterations=200)
som.save("som_model_logged.npz")

print("\nA bad input is logged, then raised:")
try:
    SOM(width=2, height=2)
except ValueError as e:
    print("caller still sees:", e)

2026-10-30 11:53:14,461 INFO     som: fit started: grid=10x10 features=3 samples=200 iterations=200


2026-10-30 11:53:14,461 DEBUG    som: iteration=0 radius=5.000 learning_rate=0.1000


2026-10-30 11:53:14,506 DEBUG    som: iteration=20 radius=4.257 learning_rate=0.0851


2026-10-30 11:53:14,552 DEBUG    som: iteration=40 radius=3.624 learning_rate=0.0725


2026-10-30 11:53:14,599 DEBUG    som: iteration=60 radius=3.085 learning_rate=0.0617


...

2026-10-30 11:53:14,873 DEBUG    som: iteration=180 radius=1.175 learning_rate=0.0235


2026-10-30 11:53:14,920 INFO     som: fit finished in 0.46s: quantisation_error=0.1366


2026-10-30 11:53:14,921 INFO     som: model saved to som_model_logged.npz



A bad input is logged, then raised:
2026-10-30 11:53:14,922 ERROR    som: Invalid grid 2x2: largest side must be > 2


caller still sees: Largest grid side must be > 2, otherwise the radius cannot decay


## Step 3: Structured logs for a log store

**Problem**

The lines above are for a human reading a terminal. In production logs go to a store such as CloudWatch Logs, Datadog or an ELK stack, and people search them by field: every `fit` slower than 60 seconds, every error for one request id. A free-text line has no fields.

**Fix**

One JSON object per line. Fixed fields (`time`, `level`, `logger`, `message`) plus whatever the call attaches through `extra=`. The code that logs does not change. Only the formatter does, which is the point of separating the two.

**Result**

The same `fit` call now produces lines a log store can index.

In [5]:
import json
from datetime import datetime, timezone


class JsonFormatter(logging.Formatter):
    """One JSON object per log line. Extra fields passed with extra={} are included."""
    STANDARD = set(vars(logging.makeLogRecord({})))

    def format(self, record):
        entry = {
            "time": datetime.fromtimestamp(record.created, tz=timezone.utc).isoformat(timespec="milliseconds"),
            "level": record.levelname,
            "logger": record.name,
            "message": record.getMessage(),
        }
        for k, v in vars(record).items():          # anything the caller attached with extra=
            if k not in self.STANDARD and not k.startswith("_"):
                entry[k] = v
        if record.exc_info:
            entry["exception"] = self.formatException(record.exc_info)
        return json.dumps(entry, default=str)


reset_handlers()
json_console = logging.StreamHandler(sys.stdout)
json_console.setLevel(logging.INFO)               # DEBUG stays off in production
json_console.setFormatter(JsonFormatter())
logger.addHandler(json_console)

som = SOM(width=10, height=10, seed=42).fit(data, n_iterations=200)
logger.info("batch scored", extra={"request_id": "demo-001", "n_rows": 10, "latency_ms": 3.2})

{"time": "2026-10-30T03:53:14.925+00:00", "level": "INFO", "logger": "som", "message": "fit started: grid=10x10 features=3 samples=200 iterations=200"}


{"time": "2026-10-30T03:53:15.375+00:00", "level": "INFO", "logger": "som", "message": "fit finished in 0.45s: quantisation_error=0.1366"}


{"time": "2026-10-30T03:53:15.376+00:00", "level": "INFO", "logger": "som", "message": "batch scored", "request_id": "demo-001", "n_rows": 10, "latency_ms": 3.2}


## Step 4: Alerting: errors reach a person

**Problem**

A log that nobody reads is a print statement with extra steps. The panel's point was that the person who needs to know has to find out in time.

**Fix**

A second handler, set to `ERROR`, whose only job is to send an alert. Here it prints a line starting with `ALERT`. In production the same handler posts to PagerDuty, Opsgenie, a Slack channel or an SNS topic. The code that logs does not know or care.

Severity decides the route:

| Level | Route | Time to a human |
|---|---|---|
| `WARNING` | Dashboard panel, daily digest | Next working day |
| `ERROR` | On-call chat channel | Minutes |
| `CRITICAL` | Page the on-call person | Now |

**Two rules that stop alert fatigue**

- A client sending bad input is a `WARNING`, not an `ERROR`. It is their mistake, and the service handled it. Alert on the rate of those, not on each one.
- Deduplicate. The same error a thousand times in a minute is one alert with a count, not a thousand pages.

In [6]:
def send_alert(record):
    """Stand-in for PagerDuty, Opsgenie, Slack or SNS. Prints here so the notebook shows the route."""
    route = "PAGE on-call" if record.levelno >= logging.CRITICAL else "POST to on-call channel"
    print(f"ALERT [{record.levelname}] -> {route}: {record.getMessage()}")


class AlertHandler(logging.Handler):
    """Routes ERROR and above to send_alert. Deduplicates repeats within a window."""

    def __init__(self, window_seconds=60):
        super().__init__(level=logging.ERROR)
        self.window = window_seconds
        self.last_sent = {}

    def emit(self, record):
        key = (record.levelno, record.getMessage())
        now = time.time()
        if now - self.last_sent.get(key, 0) < self.window:
            return                                  # same alert inside the window: suppressed
        self.last_sent[key] = now
        send_alert(record)


alerts = AlertHandler()
logger.addHandler(alerts)

logger.warning("quantisation error 0.031 is above threshold 0.024")   # dashboard only, no alert
logger.error("prediction failed: feature count mismatch")            # alert
logger.error("prediction failed: feature count mismatch")            # same message, suppressed
logger.critical("model file som_model_logged.npz not found at startup")      # page

{"time": "2026-10-30T03:53:15.379+00:00", "level": "WARNING", "logger": "som", "message": "quantisation error 0.031 is above threshold 0.024"}


{"time": "2026-10-30T03:53:15.379+00:00", "level": "ERROR", "logger": "som", "message": "prediction failed: feature count mismatch"}


ALERT [ERROR] -> POST to on-call channel: prediction failed: feature count mismatch
{"time": "2026-10-30T03:53:15.379+00:00", "level": "ERROR", "logger": "som", "message": "prediction failed: feature count mismatch"}


{"time": "2026-10-30T03:53:15.380+00:00", "level": "CRITICAL", "logger": "som", "message": "model file som_model_logged.npz not found at startup"}


ALERT [CRITICAL] -> PAGE on-call: model file som_model_logged.npz not found at startup


## Step 5: Health check with quantisation error

**Problem**

A service can be up, fast and wrong. If the data coming in has drifted away from the data the map was trained on, every request still returns a grid position. Nothing fails, and the answers are meaningless.

**Fix**

The map already has a quality measure: quantisation error, the average distance from an input to its best matching node. Score each batch of new data the same way and compare it with a reference. Above a threshold, log a `WARNING` with both numbers, so the drift shows on a dashboard and someone decides whether to retrain.

**Which reference**

The error on the training data is optimistic, since the map was fitted to those exact points. The reference is the error on the 50 held-out points the map never saw. The threshold here is twice that. In a real system the factor comes from watching the number for a few weeks.

**Result**

Data from the training distribution passes. Data shifted outside the unit cube trips the warning.

In [7]:
som.reference_error_ = som.quantisation_error(held_out)
logger.info("reference quantisation error on held-out data: %.4f (training data: %.4f)",
            som.reference_error_, som.train_error_)


def health_check(som, batch, factor=2.0):
    """Compare the batch's quantisation error with the held-out reference. Warn above factor x reference."""
    qe = som.quantisation_error(batch)
    threshold = factor * som.reference_error_
    status = "ok" if qe <= threshold else "drift"
    level = logging.INFO if status == "ok" else logging.WARNING
    logger.log(level, "health check: status=%s", status,
               extra={"quantisation_error": round(float(qe), 4), "threshold": round(float(threshold), 4)})
    return status, qe


rng = np.random.default_rng(7)
in_distribution = rng.random((20, 3))             # same range as the training data
shifted = rng.random((20, 3)) + 0.8               # drifted outside the training range

health_check(som, in_distribution)
health_check(som, shifted);

{"time": "2026-10-30T03:53:15.383+00:00", "level": "INFO", "logger": "som", "message": "reference quantisation error on held-out data: 0.1477 (training data: 0.1366)"}


{"time": "2026-10-30T03:53:15.384+00:00", "level": "INFO", "logger": "som", "message": "health check: status=ok", "quantisation_error": 0.1677, "threshold": 0.2954}


{"time": "2026-10-30T03:53:15.384+00:00", "level": "WARNING", "logger": "som", "message": "health check: status=drift", "quantisation_error": 0.9761, "threshold": 0.2954}


## Step 6: A FastAPI endpoint with request logging

**Problem**

This was the panel's question: the map runs behind a website. Each request needs a record of what came in, how long it took and how it ended, and each failure needs to be sorted into the client's fault or ours.

**Fix**

A small FastAPI app around the saved model.

- **Startup** (the `lifespan` handler) loads `som_model_logged.npz`. If that fails it is `CRITICAL` and the service does not start.
- **Middleware** gives every request an id, times it and logs one `INFO` line: id, path, status, latency. A slow request is a `WARNING`.
- **`/predict`** validates the payload. A wrong shape returns `422` and logs a `WARNING` with the request id: the client's fault, handled, no alert.
- **Anything unexpected** returns `500` and logs `ERROR` with the stack trace through `logger.exception`, which the alert handler sends on. The client gets the request id to quote, not the stack trace.
- **`/health`** returns the status and the last quantisation error, so a load balancer or a monitor can poll it.

The app runs in-process here through `TestClient`, so nothing listens on a port.

In [8]:
import uuid
from contextlib import asynccontextmanager
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

MODEL_PATH = "som_model_logged.npz"
SLOW_MS = 200

state = {"model": None, "last_health": None}


@asynccontextmanager
async def lifespan(app):
    try:
        model = SOM.load(MODEL_PATH)
        model.reference_error_ = model.quantisation_error(held_out)   # stored beside the model in a real build
        state["model"] = model
    except Exception:
        logger.critical("cannot load model from %s, service must not start", MODEL_PATH, exc_info=True)
        raise
    yield
    logger.info("service shutting down")


app = FastAPI(title="SOM service", lifespan=lifespan)


@app.middleware("http")
async def log_requests(request: Request, call_next):
    request_id = str(uuid.uuid4())[:8]
    request.state.request_id = request_id
    start = time.perf_counter()
    response = await call_next(request)
    latency_ms = (time.perf_counter() - start) * 1000
    level = logging.WARNING if latency_ms > SLOW_MS else logging.INFO
    logger.log(level, "request %s %s -> %s", request.method, request.url.path, response.status_code,
               extra={"request_id": request_id, "latency_ms": round(latency_ms, 1)})
    response.headers["X-Request-ID"] = request_id
    return response


@app.exception_handler(Exception)
async def unexpected_error(request: Request, exc: Exception):
    request_id = getattr(request.state, "request_id", "unknown")
    logger.exception("unhandled error on %s", request.url.path, extra={"request_id": request_id})
    return JSONResponse(status_code=500, content={"error": "internal error", "request_id": request_id})


class PredictRequest(BaseModel):
    rows: list[list[float]] = Field(..., min_length=1, description="n_samples x n_features")


@app.post("/predict")
def predict(req: PredictRequest):
    model = state["model"]
    X = np.array(req.rows, dtype=float)
    if X.ndim != 2 or X.shape[1] != model.weights.shape[2]:
        logger.warning("client sent %s, model expects n x %s", X.shape, model.weights.shape[2])
        raise HTTPException(status_code=422, detail=f"each row must have {model.weights.shape[2]} features")
    positions = model.predict(X)
    status, qe = health_check(model, X)
    state["last_health"] = {"status": status, "quantisation_error": round(float(qe), 4)}
    return {"positions": positions.tolist(), "health": state["last_health"]}


@app.get("/health")
def health():
    return {"status": "ok" if state["model"] is not None else "no model", "last_batch": state["last_health"]}

In [9]:
client = TestClient(app, raise_server_exceptions=False)

with client:
    print("--- good request ---")
    r = client.post("/predict", json={"rows": in_distribution[:3].tolist()})
    print(r.status_code, r.json()["health"], "request id", r.headers["X-Request-ID"])

    print("\n--- client error: wrong number of features (422, WARNING, no alert) ---")
    r = client.post("/predict", json={"rows": [[0.1, 0.2]]})
    print(r.status_code, r.json())

    print("\n--- drifted batch (200, health WARNING) ---")
    r = client.post("/predict", json={"rows": shifted[:3].tolist()})
    print(r.status_code, r.json()["health"])

    print("\n--- our fault: model removed under the service (500, ERROR with stack trace, alert) ---")
    saved = state["model"].weights
    state["model"].weights = None
    r = client.post("/predict", json={"rows": in_distribution[:3].tolist()})
    print(r.status_code, r.json())
    state["model"].weights = saved

    print("\n--- health endpoint ---")
    print(client.get("/health").json())

{"time": "2026-10-30T03:53:15.484+00:00", "level": "INFO", "logger": "som", "message": "model loaded from som_model_logged.npz"}


--- good request ---
{"time": "2026-10-30T03:53:15.487+00:00", "level": "INFO", "logger": "som", "message": "health check: status=ok", "quantisation_error": 0.1599, "threshold": 0.2954}


{"time": "2026-10-30T03:53:15.487+00:00", "level": "INFO", "logger": "som", "message": "request POST /predict -> 200", "request_id": "015e96ce", "latency_ms": 1.2}


200 {'status': 'ok', 'quantisation_error': 0.1599} request id 015e96ce

--- client error: wrong number of features (422, WARNING, no alert) ---
{"time": "2026-10-30T03:53:15.489+00:00", "level": "WARNING", "logger": "som", "message": "client sent (1, 2), model expects n x 3"}


{"time": "2026-10-30T03:53:15.489+00:00", "level": "INFO", "logger": "som", "message": "request POST /predict -> 422", "request_id": "264433c2", "latency_ms": 0.6}


422 {'detail': 'each row must have 3 features'}

--- drifted batch (200, health WARNING) ---
{"time": "2026-10-30T03:53:15.491+00:00", "level": "WARNING", "logger": "som", "message": "health check: status=drift", "quantisation_error": 0.8901, "threshold": 0.2954}


{"time": "2026-10-30T03:53:15.491+00:00", "level": "INFO", "logger": "som", "message": "request POST /predict -> 200", "request_id": "43aa795a", "latency_ms": 0.7}


200 {'status': 'drift', 'quantisation_error': 0.8901}

--- our fault: model removed under the service (500, ERROR with stack trace, alert) ---
{"time": "2026-10-30T03:53:15.493+00:00", "level": "ERROR", "logger": "som", "message": "unhandled error on /predict", "request_id": "bed6b97f", "exception": "Traceback (most recent call last):\n  File \"/opt/anaconda3/lib/python3.12/site-packages/starlette/middleware/errors.py\", line 164, in __call__\n    await self.app(scope, receive, _send)\n  File \"/opt/anaconda3/lib/python3.12/site-packages/fastapi/telemetry/_asgi.py\", line 151, in __call__\n    await self.app(scope, receive, send)\n  File \"/opt/anaconda3/lib/python3.12/site-packages/starlette/middleware/base.py\", line 203, in __call__\n    response = await self.dispatch_func(request, call_next)\n               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File \"/var/folders/n9/c91lcyc547l1m8901_vnpblw0000gn/T/ipykernel_17968/4181531467.py\", line 35, in log_requests\n    response =

ALERT [ERROR] -> POST to on-call channel: unhandled error on /predict
500 {'error': 'internal error', 'request_id': 'bed6b97f'}

--- health endpoint ---
{"time": "2026-10-30T03:53:15.495+00:00", "level": "INFO", "logger": "som", "message": "request GET /health -> 200", "request_id": "c3b8cadf", "latency_ms": 0.3}


{'status': 'ok', 'last_batch': {'status': 'drift', 'quantisation_error': 0.8901}}
{"time": "2026-10-30T03:53:15.496+00:00", "level": "INFO", "logger": "som", "message": "service shutting down"}


## Step 7: Tests that check the logging

**Problem**

Logging and alerting are code too. If a refactor removes the `ERROR` line before a raise, nothing fails and the on-call person stops hearing about it.

**Fix**

Tests that capture log records and assert on them. Under `pytest` this is the `caplog` fixture. Here a list handler does the same job so the cells run on their own. Three checks:

1. a bad input logs at `ERROR` before raising
2. a drifted batch logs at `WARNING` with both numbers attached
3. a client error returns `422` and does not reach the alert handler

In [10]:
class ListHandler(logging.Handler):
    """Collects records so tests can assert on them. pytest's caplog does this for you."""
    def __init__(self):
        super().__init__(level=logging.DEBUG)
        self.records = []
    def emit(self, record):
        self.records.append(record)


def with_capture(fn):
    cap = ListHandler()
    logger.addHandler(cap)
    try:
        fn()
    finally:
        logger.removeHandler(cap)
    return cap.records


def test_bad_input_logs_error_then_raises():
    def act():
        try:
            SOM(width=2, height=2)
        except ValueError:
            pass
    recs = with_capture(act)
    assert any(r.levelno == logging.ERROR and "Invalid grid" in r.getMessage() for r in recs)


def test_drift_logs_warning_with_numbers():
    recs = with_capture(lambda: health_check(som, shifted))
    warn = [r for r in recs if r.levelno == logging.WARNING]
    assert warn and hasattr(warn[0], "quantisation_error") and hasattr(warn[0], "threshold")


def test_client_error_is_422_and_not_an_alert():
    sent_before = dict(alerts.last_sent)
    with client:
        r = client.post("/predict", json={"rows": [[0.1, 0.2]]})
    assert r.status_code == 422
    assert alerts.last_sent == sent_before          # nothing new reached the alert handler


json_console.setLevel(logging.CRITICAL)             # quiet the console while the tests run
for test in (test_bad_input_logs_error_then_raises, test_drift_logs_warning_with_numbers, test_client_error_is_422_and_not_an_alert):
    test()
    print("PASS", test.__name__)
json_console.setLevel(logging.INFO)

ALERT [ERROR] -> POST to on-call channel: Invalid grid 2x2: largest side must be > 2
PASS test_bad_input_logs_error_then_raises
PASS test_drift_logs_warning_with_numbers
PASS test_client_error_is_422_and_not_an_alert


## Step 8: What this looks like in production

Nothing in the code changes. The handlers change.

| Piece in this notebook | In production |
|---|---|
| `StreamHandler` to stdout with `JsonFormatter` | The container writes JSON to stdout. CloudWatch Logs, Datadog or Fluent Bit collect it. Fields are searchable |
| `AlertHandler` calling `send_alert` | A CloudWatch metric filter on `level=ERROR` feeding an alarm and an SNS topic, or a Sentry or Datadog monitor, into PagerDuty or Slack |
| `health_check` logging quantisation error | The same number published as a custom metric. A dashboard panel, an alarm on the threshold, and a retrain job triggered when it fires |
| The request middleware | p50 and p95 latency, request count and error rate as metrics. Alarm on error rate above a percentage, not on single errors |
| `/health` | The load balancer's health check and the deployment's readiness probe |
| The tests | Run in CI on every change, so a removed log line fails the build |

One more piece that is not code: a runbook. Each alert names the person who owns it, what to check first and how to roll back. An alert with no runbook is noise by the second week.

**The answer to the panel's question in three sentences.** Every event is logged with a level and the request id, as JSON, to a log store. Errors and above route to a person through an alert handler with deduplication, while client mistakes stay at warning. The service reports its own quality through quantisation error on live data, so drift is caught before anyone complains.